In [1]:
import numpy as np

In [ ]:
import cv2
print(cv2.__version__)
print(cv2.face.LBPHFaceRecognizer_create())

In [5]:
pip install opencv-python


Note: you may need to restart the kernel to use updated packages.


You should consider upgrading via the 'C:\Users\AarushiGarg\AppData\Local\Programs\Python\Python39\python.exe -m pip install --upgrade pip' command.


In [3]:
import cv2
import os
import time
import numpy as np  # Make sure this is imported

face_id = 1  
save_path = f'dataset/{face_id}'
os.makedirs(save_path, exist_ok=True)

# Load DNN face detector
prototxt_path = "deploy.prototxt"
caffemodel_path = "res10_300x300_ssd_iter_140000.caffemodel"
detector = cv2.dnn.readNetFromCaffe(prototxt_path, caffemodel_path)

cap = cv2.VideoCapture(0)
count = 0
last_saved_time = time.time()
save_interval = 1.5  # seconds between saves

while True:
    ret, frame = cap.read()
    if not ret:
        break
        
    (h, w) = frame.shape[:2]
    
    blob = cv2.dnn.blobFromImage(
        cv2.resize(frame, (300, 300)), 
        1.0, 
        (300, 300),
        (104.0, 177.0, 123.0)
    )
    detector.setInput(blob)
    detections = detector.forward()

    for i in range(0, detections.shape[2]):
        confidence = detections[0, 0, i, 2]
        
        if confidence > 0.7:
            box = detections[0, 0, i, 3:7] * np.array([w, h, w, h])
            (startX, startY, endX, endY) = box.astype("int")

            startX = max(0, startX)
            startY = max(0, startY)
            endX = min(w, endX)
            endY = min(h, endY)

            face = frame[startY:endY, startX:endX]
            gray_face = cv2.cvtColor(face, cv2.COLOR_BGR2GRAY)

            # Save only if interval time passed
            if time.time() - last_saved_time >= save_interval:
                count += 1
                cv2.imwrite(f"{save_path}/{count}.jpg", gray_face)
                last_saved_time = time.time()

            cv2.rectangle(frame, (startX, startY), (endX, endY), (255, 0, 0), 2)

    cv2.imshow('Capturing Faces - Press Q to Stop', frame)

    if cv2.waitKey(1) & 0xFF == ord('q') or count >= 30:
        break

cap.release()
cv2.destroyAllWindows()
print(f"[INFO] Saved {count} images to {save_path}")


[INFO] Saved 12 images to dataset/1


In [ ]:
import cv2
import os
import numpy as np

recognizer = cv2.face.LBPHFaceRecognizer_create()
prototxt_path = "deploy.prototxt"
caffemodel_path = "res10_300x300_ssd_iter_140000.caffemodel"
detector = cv2.dnn.readNetFromCaffe(prototxt_path, caffemodel_path)

def get_images_and_labels(dataset_path):
    images = []
    labels = []
    
    # Get only numeric folder names (person IDs)
    valid_folders = [f for f in os.listdir(dataset_path) 
                    if os.path.isdir(os.path.join(dataset_path, f)) 
                    and f.isdigit()]
    
    for label_folder in valid_folders:
        label = int(label_folder)
        label_path = os.path.join(dataset_path, label_folder)
        
        for image_name in os.listdir(label_path):
            img_path = os.path.join(label_path, image_name)
            # Skip hidden files (like .DS_Store on Mac)
            if image_name.startswith('.'):
                continue
                
            img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
            if img is not None:
                labels.append(label)
                images.append(img)

    return images, labels

# Training
faces, ids = get_images_and_labels('dataset')

if len(faces) == 0:
    print("[ERROR] No faces found in dataset directory")
    print("Make sure:")
    print("1. The 'dataset' folder exists")
    print("2. It contains subfolders with numeric names (like '1', '2')")
    print("3. Each subfolder contains face images")
else:
    recognizer.train(faces, np.array(ids))
    recognizer.save('trainer.yml')
    print(f"[SUCCESS] Trained on {len(faces)} images from {len(set(ids))} persons")
    print("Model saved to trainer.yml")

In [5]:
import cv2
import numpy as np
from matplotlib import pyplot as plt
%matplotlib inline

import smtplib
from email.mime.text import MIMEText
from email.mime.image import MIMEImage
from email.mime.multipart import MIMEMultipart
from IPython.display import display, clear_output
from io import BytesIO

In [6]:
prototxt_path = "deploy.prototxt"
caffemodel_path = "res10_300x300_ssd_iter_140000.caffemodel"
detector = cv2.dnn.readNetFromCaffe(prototxt_path, caffemodel_path)
recognizer = cv2.face.LBPHFaceRecognizer_create()
recognizer.read('trainer.yml')

In [7]:
authorized_ids = [1]  # Add more authorized IDs as needed
min_area = 500
threshold_value = 25
blur_value = (21, 21)
email_sent = False

fgbg = cv2.createBackgroundSubtractorMOG2(history=500, varThreshold=16, detectShadows=True)

In [8]:
import smtplib
from email.mime.text import MIMEText
from email.mime.image import MIMEImage
from email.mime.multipart import MIMEMultipart

def send_alert_email(frame):
    msg = MIMEMultipart()
    msg['Subject'] = '🚨 INTRUSION ALERT!'
    msg['From'] = 'aarushigarg16@yahoo.com'
    msg['To'] = 'aarushigarg8765@gmail.com'

    msg.attach(MIMEText("An unauthorized person was detected!"))

    _, img_encoded = cv2.imencode('.jpg', frame)
    img_part = MIMEImage(img_encoded.tobytes())
    msg.attach(img_part)

    app_password = "llrxtnswijaymacl"  

    try:
        with smtplib.SMTP_SSL('smtp.mail.yahoo.com', 465) as server:
            server.login('aarushigarg16@yahoo.com', app_password)
            server.send_message(msg)
        print("✅ Email sent successfully.")
    except Exception as e:
        print(f"❌ Failed to send email: {e}")

In [9]:
def show_frame(frame):
    clear_output(wait=True)
    plt.figure(figsize=(10, 6))
    plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    plt.axis('off')
    plt.show()

In [10]:
cap = cv2.VideoCapture(0)

In [ ]:
while True:
    ret, frame = cap.read()
    if not ret:
        break

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    gray_blur = cv2.GaussianBlur(gray, blur_value, 0)
    fgmask = fgbg.apply(frame)

    _, thresh = cv2.threshold(fgmask, threshold_value, 255, cv2.THRESH_BINARY)
    contours, _ = cv2.findContours(thresh.copy(), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    intrusion_detected = False
    authorized_detected = False

    # DNN-based face detection
    blob = cv2.dnn.blobFromImage(frame, 1.0, (300, 300), (104.0, 177.0, 123.0))
    detector.setInput(blob)
    detections = detector.forward()

    for i in range(detections.shape[2]):
        confidence = detections[0, 0, i, 2]
        if confidence > 0.7:
            box = detections[0, 0, i, 3:7] * np.array([frame.shape[1], frame.shape[0], frame.shape[1], frame.shape[0]])
            (x, y, w, h) = box.astype("int")
            face_roi = gray[y:h, x:w]
            face_roi = cv2.resize(face_roi, (200, 200))

            id_, conf = recognizer.predict(face_roi)
            if id_ in authorized_ids and conf < 70:
                authorized_detected = True
                cv2.putText(frame, f"Authorized (Conf: {conf:.2f})", (x, y-10),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
            else:
                intrusion_detected = True
                cv2.rectangle(frame, (x, y), (w, h), (0, 0, 255), 2)
                cv2.putText(frame, "INTRUDER", (x, y-30),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)

    if intrusion_detected and not email_sent:
        send_alert_email(frame)
        email_sent = True

    show_frame(frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break